# 06 — Naive seasonal baselines

Evaluate same-hour previous-hour/day/week forecasts on fixed Jan 21–25 validation and Jan 26–31 test windows; inspect demand categories using TRAINING data only.

**Execution status:** Code is provided for reproducibility. See `results/observed_results.json` for results independently reported from an earlier Fabric run; this generated notebook has not been executed in this environment.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
gold=spark.table("gold_hourly_zone_demand")
w=Window.partitionBy("PULocationID").orderBy("pickup_hour")
base=(gold.withColumn("prediction_lag_1",F.lag("pickup_count",1).over(w))
 .withColumn("prediction_lag_24",F.lag("pickup_count",24).over(w))
 .withColumn("prediction_lag_168",F.lag("pickup_count",168).over(w)))
valid=base.filter((F.col("pickup_hour")>="2025-01-21")&(F.col("pickup_hour")<"2025-01-26"))
test=base.filter((F.col("pickup_hour")>="2025-01-26")&(F.col("pickup_hour")<"2025-02-01"))
assert valid.count()==31080 and test.count()==37296

In [ ]:
def metric_summary(frame):
    return frame.agg(*[F.round(F.avg(F.abs(F.col("pickup_count")-F.col(f"prediction_lag_{lag}"))),4).alias(f"mae_lag_{lag}") for lag in (1,24,168)])
print("VALIDATION (model selection only)");metric_summary(valid).show()
print("TEST (previously inspected; not pristine holdout)");metric_summary(test).show()

In [ ]:
zone_training=(gold.filter(F.col("pickup_hour")<"2025-01-21")
 .groupBy("PULocationID").agg(F.avg("pickup_count").alias("training_avg_pickups"))
 .withColumn("demand_category",F.when(F.col("training_avg_pickups")<1,"Very Low").when(F.col("training_avg_pickups")<10,"Low").when(F.col("training_avg_pickups")<50,"Medium").otherwise("High")))
validation_category=valid.join(zone_training.select("PULocationID","demand_category"),"PULocationID","left")
validation_category.groupBy("demand_category").agg(F.count("*").alias("observations"),*[F.round(F.avg(F.abs(F.col("pickup_count")-F.col(f"prediction_lag_{k}"))),4).alias(f"mae_lag_{k}") for k in (1,24,168)]).show()

**Important evaluation limitation:** Test results were inspected in the original learning conversation before the final experiment; this test is **not** an untouched holdout. Selecting models must use the validation window. Re-evaluate later on a new month for credible final out-of-time performance.